# IN4640 Assignment 2
## Fitting and Alignment

### Problem 1: Total Least Squares and RANSAC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import RANSACRegressor

# Load dataset
D = np.genfromtxt("lines.csv", delimiter=",", skip_header=1)

# --- Part (a): Total Least Squares (TLS) on the first line ---
x1 = D[:, 0]
y1 = D[:, 3]

# Centering the data
x1_mean, y1_mean = np.mean(x1), np.mean(y1)
centered_data = np.vstack((x1 - x1_mean, y1 - y1_mean)).T

# Compute SVD
U, S, Vt = np.linalg.svd(centered_data)

# The normal vector of the best fit line is the right singular vector with the smallest singular value
a, b = Vt[-1]
c = -(a * x1_mean + b * y1_mean)

print("--- 1(a) Total Least Squares ---")
print(f"Line equation: {a:.4f}x + {b:.4f}y + {c:.4f} = 0\n")

# --- Part (b): RANSAC to find three lines ---
X_cols = D[:, :3]
Y_cols = D[:, 3:]
X_all = X_cols.flatten().reshape(-1, 1)
Y_all = Y_cols.flatten()

points_X = X_all.copy()
points_Y = Y_all.copy()

plt.figure(figsize=(10, 6))
plt.scatter(points_X, points_Y, color='lightgray', label='All Unfitted Points', alpha=0.6)

colors = ['red', 'green', 'blue']

print("--- 1(b) RANSAC Line Parameters ---")
for i in range(3):
    if len(points_X) < 2:
        break
        
    ransac = RANSACRegressor(min_samples=2, residual_threshold=0.5, max_trials=1000)
    ransac.fit(points_X, points_Y)
    
    inlier_mask = ransac.inlier_mask_
    outlier_mask = np.logical_not(inlier_mask)
    
    # Extract line parameters (y = mx + c)
    m = ransac.estimator_.coef_[0]
    c_int = ransac.estimator_.intercept_
    print(f"Line {i+1}: y = {m:.4f}x + {c_int:.4f} (Inliers: {np.sum(inlier_mask)})")
    
    # Plot inliers
    plt.scatter(points_X[inlier_mask], points_Y[inlier_mask], color=colors[i], label=f'Line {i+1} Inliers')
    
    # Plot the best fit line
    x_min, x_max = points_X[inlier_mask].min(), points_X[inlier_mask].max()
    line_x = np.array([[x_min], [x_max]])
    line_y = ransac.predict(line_x)
    plt.plot(line_x, line_y, color=colors[i], linewidth=2)
    
    # Mask consensus (remove inliers for the next iteration)
    points_X = points_X[outlier_mask]
    points_Y = points_Y[outlier_mask]

plt.legend()
plt.title('RANSAC Multiple Line Fitting')
plt.xlabel('X')
plt.ylabel('Y')
plt.grid(True)
plt.show()


### Problem 2: Camera Geometry and Earring Size

In [ ]:
import cv2

# Given Parameters
f = 8.0  # Focal length in mm
Z = 720.0  # Distance to imaging plane in mm
pixel_size = 0.0022  # Pixel size in mm (2.2 micrometers)

# Load the image
img = cv2.imread("earrings.jpg", cv2.IMREAD_GRAYSCALE)

if img is None:
    print("Error: 'earrings.jpg' not found. Please ensure it is in the same directory.")
else:
    # Thresholding to segment the earrings from the background
    _, thresh = cv2.threshold(img, 128, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    
    # Find contours
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Sort contours by area to find the two largest ones (the earrings)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:2]
    
    print("--- Earring Real-World Sizes ---")
    for i, cnt in enumerate(contours):
        # Get bounding box for each earring
        x_px, y_px, w_px, h_px = cv2.boundingRect(cnt)
        
        # Calculate physical dimensions
        width_mm = (w_px * pixel_size * Z) / f
        height_mm = (h_px * pixel_size * Z) / f
        
        print(f"Earring {i+1}:")
        print(f"  Pixel Bounding Box: {w_px}px width x {h_px}px height")
        print(f"  Real-World Size: {width_mm:.2f}mm width x {height_mm:.2f}mm height\n")


### Problem 3: Homography and Image Superimposition
*Note: Ensure you run this notebook locally, as it requires opening a GUI window for point selection.*

In [ ]:
import cv2
import numpy as np
import urllib.request

points = []

def mouse_callback(event, x, y, flags, param):
    global points, img_display
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(points) < 4:
            points.append((x, y))
            print(f"Point {len(points)}: ({x}, {y})")
            cv2.circle(img_display, (x, y), 5, (0, 0, 255), -1)
            cv2.imshow("Turf", img_display)

# Load the turf image
img_turf = cv2.imread("turf.jpg")
if img_turf is None:
    print("Error: 'turf.jpg' not found. Please ensure it is in the same directory.")
else:
    # Load a sample flag (e.g., Sri Lanka flag from Wikimedia)
    url = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/11/Flag_of_Sri_Lanka.svg/800px-Flag_of_Sri_Lanka.svg.png"
    req = urllib.request.urlopen(url)
    arr = np.asarray(bytearray(req.read()), dtype=np.uint8)
    img_flag = cv2.imdecode(arr, -1)
    
    # Drop alpha channel if it exists
    if img_flag.shape[2] == 4:
        img_flag = img_flag[:, :, :3]

    img_display = img_turf.copy()

    cv2.namedWindow("Turf")
    cv2.setMouseCallback("Turf", mouse_callback)

    print("Please click exactly 4 points on the turf in a clockwise direction, starting from top-left.")
    print("Press any key to execute after making the 4 selections.")
    cv2.imshow("Turf", img_display)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    if len(points) == 4:
        pts_dst = np.array(points, dtype=np.float32)
        
        # Source points (corners of the flag)
        h_flag, w_flag = img_flag.shape[:2]
        pts_src = np.array([
            [0, 0],
            [w_flag - 1, 0],
            [w_flag - 1, h_flag - 1],
            [0, h_flag - 1]
        ], dtype=np.float32)
        
        # 1. Find the homography matrix
        H, status = cv2.findHomography(pts_src, pts_dst)
        
        # 2. Warp the flag onto the turf
        warped_flag = cv2.warpPerspective(img_flag, H, (img_turf.shape[1], img_turf.shape[0]))
        
        # 3. Create a mask and blend images
        mask = np.zeros(img_turf.shape, dtype=np.uint8)
        cv2.fillConvexPoly(mask, pts_dst.astype(np.int32), (255, 255, 255))
        
        # Invert mask for the background
        mask_inv = cv2.bitwise_not(mask)
        turf_bg = cv2.bitwise_and(img_turf, mask_inv)
        
        # Combine background and warped flag
        final_result = cv2.bitwise_or(turf_bg, warped_flag)
        
        cv2.imshow("Final Result", final_result)
        cv2.imwrite("superimposed_turf.jpg", final_result)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    else:
        print("Error: You did not select exactly 4 points.")
